# Sleep Quality Causal Analysis

## Phase 1: Authenticate with Whoop

Run the cell below. It will:
1. Open your browser to Whoop's OAuth consent page
2. Capture the authorization code via a local server on port 9876
3. Exchange it for access + refresh tokens
4. Save tokens to `.env` for future runs

**Prerequisites:**
- In your Whoop dev app, register `http://localhost:9876/callback` as an allowed redirect URI (exact match, including port).
- Fill `WHOOP_CLIENT_ID` and `WHOOP_CLIENT_SECRET` in `.env`.


In [ ]:
import os
import json
import time
import secrets
import threading
import webbrowser
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.parse import urlparse, parse_qs
from pathlib import Path

import httpx
from dotenv import load_dotenv, set_key

load_dotenv()

CLIENT_ID = os.environ.get("WHOOP_CLIENT_ID", "")
CLIENT_SECRET = os.environ.get("WHOOP_CLIENT_SECRET", "")
REDIRECT_URI = "http://localhost:9876/callback"
AUTH_URL = "https://api.prod.whoop.com/oauth/oauth2/auth"
TOKEN_URL = "https://api.prod.whoop.com/oauth/oauth2/token"
SCOPES = "read:recovery read:cycles read:sleep read:workout read:profile read:body_measurement"
ENV_PATH = Path("../.env")

assert CLIENT_ID, "Set WHOOP_CLIENT_ID in .env"
assert CLIENT_SECRET, "Set WHOOP_CLIENT_SECRET in .env"

# Whoop requires an opaque `state` param on the authorize request; it is
# echoed back on the callback and must match (CSRF protection).
STATE = secrets.token_urlsafe(32)

_auth_code = None
_state_ok = False

class _CallbackHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        global _auth_code, _state_ok
        qs = parse_qs(urlparse(self.path).query)
        returned_state = qs.get("state", [None])[0]
        _state_ok = (returned_state == STATE)
        if _state_ok:
            _auth_code = qs.get("code", [None])[0]
        self.send_response(200)
        self.send_header("Content-Type", "text/html")
        self.end_headers()
        if _state_ok:
            self.wfile.write(b"<h2>Done! Close this tab and return to the notebook.</h2>")
        else:
            self.wfile.write(b"<h2>State mismatch -- possible CSRF. Re-run the cell.</h2>")

    def log_message(self, *args):
        pass

server = HTTPServer(("localhost", 9876), _CallbackHandler)
thread = threading.Thread(target=server.handle_request, daemon=True)
thread.start()

auth_link = (
    f"{AUTH_URL}?client_id={CLIENT_ID}"
    f"&redirect_uri={REDIRECT_URI}"
    f"&response_type=code"
    f"&state={STATE}"
    f"&scope={SCOPES.replace(' ', '%20')}"
)
print(f"Opening browser for Whoop login...\n{auth_link}")
webbrowser.open(auth_link)

thread.join(timeout=120)
server.server_close()

assert _state_ok, "State mismatch or no callback received -- re-run this cell."
assert _auth_code, "No auth code received -- did you complete login?"

resp = httpx.post(TOKEN_URL, data={
    "grant_type": "authorization_code",
    "code": _auth_code,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "redirect_uri": REDIRECT_URI,
})
resp.raise_for_status()
tokens = resp.json()

ACCESS_TOKEN = tokens["access_token"]
REFRESH_TOKEN = tokens["refresh_token"]
EXPIRES_IN = tokens["expires_in"]

set_key(str(ENV_PATH), "WHOOP_REFRESH_TOKEN", REFRESH_TOKEN)
set_key(str(ENV_PATH), "WHOOP_ACCESS_TOKEN", ACCESS_TOKEN)

print(f"Authenticated! Token expires in {EXPIRES_IN // 3600}h.")
print(f"Refresh token saved to {ENV_PATH.resolve()}")


In [ ]:
def get_valid_token() -> str:
    """Use stored refresh token if access token expired."""
    load_dotenv(override=True)
    access_token = os.environ.get("WHOOP_ACCESS_TOKEN", "")
    refresh_token = os.environ.get("WHOOP_REFRESH_TOKEN", "")

    if access_token:
        r = httpx.get(
            "https://api.prod.whoop.com/developer/v2/user/profile/basic",
            headers={"Authorization": f"Bearer {access_token}"},
            timeout=10,
        )
        if r.status_code == 200:
            return access_token

    assert refresh_token, "No refresh token -- run the OAuth cell above first"
    resp = httpx.post(TOKEN_URL, data={
        "grant_type": "refresh_token",
        "refresh_token": refresh_token,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    })
    resp.raise_for_status()
    new_tokens = resp.json()
    set_key(str(ENV_PATH), "WHOOP_ACCESS_TOKEN", new_tokens["access_token"])
    set_key(str(ENV_PATH), "WHOOP_REFRESH_TOKEN", new_tokens["refresh_token"])
    return new_tokens["access_token"]

ACCESS_TOKEN = get_valid_token()
print("Token valid. Ready to fetch data.")


## Phase 2: Fetch & Save Data

Pulls 6 months of sleep, recovery, cycle, and workout data. Saves raw Parquet files to `data/raw/` for offline re-analysis.


In [ ]:
from datetime import date, timedelta

import pandas as pd

API_BASE = "https://api.prod.whoop.com/developer"
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

def fetch_paginated(endpoint: str, params: dict = None) -> list[dict]:
    """Fetch all pages from a Whoop v2 endpoint."""
    params = params or {}
    records = []
    next_token = None

    while True:
        req_params = {**params}
        if next_token:
            req_params["nextToken"] = next_token

        r = httpx.get(f"{API_BASE}{endpoint}", params=req_params, headers=HEADERS, timeout=30)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "2"))
            print(f"  Rate limited, waiting {wait}s...")
            time.sleep(wait)
            continue

        r.raise_for_status()
        body = r.json()
        batch = body.get("records", [])
        records.extend(batch)
        next_token = body.get("next_token") or body.get("nextToken")

        if not next_token:
            break

    return records

print(f"Fetch utility ready. Will save to {DATA_DIR.resolve()}")


In [ ]:
END = date.today().isoformat() + "T23:59:59.999Z"
START = (date.today() - timedelta(days=180)).isoformat() + "T00:00:00.000Z"

print(f"Fetching 6 months: {START[:10]} to {END[:10]}")

print("Fetching sleep...")
sleep_raw = fetch_paginated("/v2/activity/sleep")
print(f"  {len(sleep_raw)} sleep records")

print("Fetching recovery...")
recovery_raw = fetch_paginated("/v2/recovery")
print(f"  {len(recovery_raw)} recovery records")

print("Fetching cycles...")
cycle_raw = fetch_paginated("/v2/cycle")
print(f"  {len(cycle_raw)} cycle records")

print("Fetching workouts...")
workout_raw = fetch_paginated("/v2/activity/workout")
print(f"  {len(workout_raw)} workout records")


In [ ]:
import sys
sys.path.insert(0, str(Path("../src").resolve()))
from whoop_analytics.api.models import SleepRecord, RecoveryRecord
from dataclasses import asdict

if sleep_raw:
    parsed_sleep = [SleepRecord.from_api(r) for r in sleep_raw]
    df_sleep = pd.DataFrame([asdict(r) for r in parsed_sleep])
    df_sleep.to_parquet(DATA_DIR / "sleep.parquet", index=False)
    print(f"Saved sleep: {df_sleep.shape}")
else:
    print("No sleep data returned")
    df_sleep = pd.DataFrame()

if recovery_raw:
    parsed_recovery = [RecoveryRecord.from_api(r) for r in recovery_raw]
    df_recovery = pd.DataFrame([asdict(r) for r in parsed_recovery])
    df_recovery.to_parquet(DATA_DIR / "recovery.parquet", index=False)
    print(f"Saved recovery: {df_recovery.shape}")
else:
    print("No recovery data returned")
    df_recovery = pd.DataFrame()

if cycle_raw:
    rows = []
    for c in cycle_raw:
        if not c.get("end"):
            continue
        score = c.get("score") or {}
        rows.append({
            "id": c["id"],
            "start": c["start"],
            "end": c["end"],
            "strain": score.get("strain"),
            "average_heart_rate": score.get("average_heart_rate"),
            "max_heart_rate": score.get("max_heart_rate"),
            "kilojoule": score.get("kilojoule"),
        })
    df_cycles = pd.DataFrame(rows)
    df_cycles.to_parquet(DATA_DIR / "cycles.parquet", index=False)
    print(f"Saved cycles: {df_cycles.shape}")
else:
    df_cycles = pd.DataFrame()

if workout_raw:
    rows = []
    for w in workout_raw:
        score = w.get("score") or {}
        rows.append({
            "id": w["id"],
            "start": w.get("start"),
            "end": w.get("end"),
            "sport_id": w.get("sport_id"),
            "strain": score.get("strain"),
            "average_heart_rate": score.get("average_heart_rate"),
            "max_heart_rate": score.get("max_heart_rate"),
            "kilojoule": score.get("kilojoule"),
        })
    df_workouts = pd.DataFrame(rows)
    df_workouts.to_parquet(DATA_DIR / "workouts.parquet", index=False)
    print(f"Saved workouts: {df_workouts.shape}")
else:
    df_workouts = pd.DataFrame()

print("\nAll data saved to Parquet. You can skip Phase 1 & 2 on subsequent runs.")


## Phase 3: Causal Analysis on Sleep Quality

### 3a. Build Daily Dataset

Merge sleep stages, recovery metrics, and daily strain into a single time-indexed DataFrame.


In [ ]:
import numpy as np

df_sleep = pd.read_parquet(DATA_DIR / "sleep.parquet")
df_recovery = pd.read_parquet(DATA_DIR / "recovery.parquet")
df_cycles = pd.read_parquet(DATA_DIR / "cycles.parquet")

df_s = df_sleep[df_sleep["nap"] == False].copy()
df_s["date"] = pd.to_datetime(df_s["end"]).dt.normalize()
sleep_cols = [
    "total_sleep_minutes", "sws_minutes", "rem_minutes",
    "light_minutes", "disturbance_count", "respiratory_rate",
    "sleep_efficiency", "sleep_debt_minutes",
]
df_s = df_s.dropna(subset=["total_sleep_minutes"])
df_s = df_s.sort_values("total_sleep_minutes", ascending=False)
df_s = df_s.groupby("date")[sleep_cols].first()

df_r = df_recovery.copy()
df_r["date"] = pd.to_datetime(df_r["created_at"]).dt.normalize()
recovery_cols = ["recovery_score", "hrv_rmssd", "resting_hr", "spo2", "skin_temp"]
df_r = df_r.groupby("date")[recovery_cols].first()

df_c = df_cycles.copy()
df_c["date"] = pd.to_datetime(df_c["end"]).dt.normalize()
cycle_cols = ["strain", "average_heart_rate", "max_heart_rate", "kilojoule"]
df_c = df_c.groupby("date")[cycle_cols].first()

daily = df_s.join(df_r, how="outer").join(df_c, how="outer")
daily = daily.sort_index()
daily = daily.dropna(axis=1, how="all")

print(f"Daily dataset: {daily.shape[0]} days, {daily.shape[1]} features")
print(f"Date range: {daily.index.min().date()} to {daily.index.max().date()}")
print(f"\nColumns: {list(daily.columns)}")
print(f"\nMissing values:\n{daily.isnull().sum()[daily.isnull().sum() > 0]}")
daily.head()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=["Total Sleep (min)", "HRV (rmssd)", "Recovery Score"])

fig.add_trace(go.Scatter(x=daily.index, y=daily.get("total_sleep_minutes"),
                         mode="lines+markers", name="Sleep", marker=dict(size=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=daily.get("hrv_rmssd"),
                         mode="lines+markers", name="HRV", marker=dict(size=3)), row=2, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=daily.get("recovery_score"),
                         mode="lines+markers", name="Recovery", marker=dict(size=3)), row=3, col=1)

fig.update_layout(height=600, title_text="Sleep Quality Metrics Over Time", showlegend=False)
fig.show()

corr = daily.corr()
fig2 = px.imshow(corr, text_auto=".2f", aspect="auto",
                 title="Feature Correlation Matrix", color_continuous_scale="RdBu_r",
                 zmin=-1, zmax=1)
fig2.update_layout(height=600, width=700)
fig2.show()

if all(c in daily.columns for c in ["sws_minutes", "rem_minutes", "light_minutes"]):
    fig3 = go.Figure()
    for col, name in [("sws_minutes", "Deep/SWS"), ("rem_minutes", "REM"), ("light_minutes", "Light")]:
        fig3.add_trace(go.Histogram(x=daily[col].dropna(), name=name, opacity=0.7))
    fig3.update_layout(barmode="overlay", title="Sleep Stage Distributions (minutes)",
                       xaxis_title="Minutes", yaxis_title="Count")
    fig3.show()


### 3b. Feature Engineering

Add lagged features (previous 1-2 days) and rolling statistics (3 and 7 day windows) to capture temporal dynamics. The causal analysis targets **sleep quality** -- specifically `total_sleep_minutes` as primary, with `sleep_efficiency` and `sws_minutes` as secondary targets.


In [ ]:
sys.path.insert(0, str(Path("../src").resolve()))
from whoop_analytics.pipeline.features import add_lag_features, add_rolling_features

TARGET = "total_sleep_minutes"
SECONDARY_TARGETS = ["sleep_efficiency", "sws_minutes", "hrv_rmssd"]

feature_cols = [c for c in daily.columns if c not in [TARGET]]

df_analysis = daily.copy()
df_analysis = add_lag_features(df_analysis, columns=feature_cols, lags=[1, 2])
df_analysis = add_rolling_features(df_analysis, columns=feature_cols, windows=[3, 7])
df_analysis = df_analysis.dropna()

print(f"Analysis dataset: {df_analysis.shape[0]} days, {df_analysis.shape[1]} features")
print(f"Target: {TARGET}")
print(f"Available for discovery: {df_analysis.shape[1] - 1} potential causes")

slim_cols = [c for c in daily.columns if c in df_analysis.columns]
df_slim = df_analysis[slim_cols].copy()
print(f"\nSlim dataset for PCMCI: {df_slim.shape}")
df_slim.head()


### 3c. Causal Discovery -- PCMCI

Uses the PC-MCI algorithm (Peter-Clark Momentary Conditional Independence) to discover time-lagged causal relationships. This goes beyond correlation by testing conditional independence -- if A->B disappears when conditioning on C, it's not a direct cause.

Parameters:
- `max_lag=5`: test causal effects up to 5 days back
- `significance_level=0.05`: links must pass at p < 0.05


In [ ]:
from tigramite import data_processing as pp
from tigramite.independence_tests.parcorr import ParCorr
from tigramite.pcmci import PCMCI
import tigramite.plotting as tp

var_names = list(df_slim.columns)
data_array = df_slim.values.astype(np.float64)
dataframe = pp.DataFrame(data_array, var_names=var_names)

MAX_LAG = 5
ALPHA = 0.05

parcorr = ParCorr(significance="analytic")
pcmci = PCMCI(dataframe=dataframe, cond_ind_test=parcorr, verbosity=0)

print(f"Running PCMCI on {len(var_names)} variables, max_lag={MAX_LAG}...")
print(f"Variables: {var_names}")
results = pcmci.run_pcmci(tau_max=MAX_LAG, pc_alpha=None)

val_matrix = results["val_matrix"]
p_matrix = results["p_matrix"]
graph = pcmci.get_graph_from_pmatrix(
    p_matrix=p_matrix,
    alpha_level=ALPHA,
    tau_min=1,
    tau_max=MAX_LAG,
)

target_idx = var_names.index(TARGET)
causal_parents = []
for source_idx in range(len(var_names)):
    for lag in range(1, MAX_LAG + 1):
        if graph[source_idx, target_idx, lag] == "-->":
            causal_parents.append({
                "source": var_names[source_idx],
                "lag": lag,
                "strength": float(val_matrix[source_idx, target_idx, lag]),
                "p_value": float(p_matrix[source_idx, target_idx, lag]),
            })

causal_parents.sort(key=lambda x: abs(x["strength"]), reverse=True)

print(f"\n{'=' * 60}")
print(f"CAUSAL PARENTS OF '{TARGET}' (p < {ALPHA})")
print(f"{'=' * 60}")
if causal_parents:
    for p in causal_parents:
        print(f"  {p['source']} (lag={p['lag']}) -> {TARGET}  "
              f"| strength={p['strength']:.3f}, p={p['p_value']:.4f}")
else:
    print("  No significant causal parents found at this threshold.")
    print("  Try increasing ALPHA or MAX_LAG, or check data quality.")


In [ ]:
tp.plot_graph(
    val_matrix=val_matrix,
    graph=graph,
    var_names=var_names,
    figsize=(12, 8),
    node_size=0.4,
)

tp.plot_time_series_graph(
    val_matrix=val_matrix,
    graph=graph,
    var_names=var_names,
    figsize=(14, 6),
)


In [ ]:
import networkx as nx

G = nx.DiGraph()
for v in var_names:
    G.add_node(v)

for p in causal_parents:
    G.add_edge(p["source"], TARGET,
               weight=abs(p["strength"]),
               lag=p["lag"])

for i in range(len(var_names)):
    for j in range(len(var_names)):
        if j == target_idx:
            continue
        for lag in range(1, MAX_LAG + 1):
            if graph[i, j, lag] == "-->":
                G.add_edge(var_names[i], var_names[j],
                           weight=abs(float(val_matrix[i, j, lag])),
                           lag=lag)

pos = nx.spring_layout(G, k=2, seed=42)

edge_x, edge_y = [], []
for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_colors = ["red" if n == TARGET else "lightblue" for n in G.nodes()]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                         line=dict(width=1, color="#888"), hoverinfo="none"))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers+text",
                         marker=dict(size=20, color=node_colors, line=dict(width=2, color="black")),
                         text=list(G.nodes()), textposition="top center",
                         hoverinfo="text"))
fig.update_layout(title=f"Causal Graph (target: {TARGET})",
                  showlegend=False, height=500,
                  xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()


### 3d. Causal Effect Estimation (DoWhy)

For each discovered causal parent, estimate the **Average Treatment Effect** using DoWhy's backdoor linear regression. Then refute using:
- Random common cause: checks if adding a random variable changes the estimate
- Placebo treatment: replaces the real cause with random noise

Only effects that survive both refutations are considered robust.


In [ ]:
from dowhy import CausalModel

def estimate_causal_effect(df, treatment, outcome, lag, common_causes=None):
    """Estimate ATE with DoWhy, including refutation tests."""
    df_est = df.copy()

    if lag > 0:
        df_est[f"{treatment}_lag{lag}"] = df_est[treatment].shift(lag)
        df_est = df_est.dropna()
        treatment_col = f"{treatment}_lag{lag}"
    else:
        treatment_col = treatment

    causes = common_causes or []
    edges = [f'"{treatment_col}" -> "{outcome}"']
    for c in causes:
        edges.append(f'"{c}" -> "{treatment_col}"')
        edges.append(f'"{c}" -> "{outcome}"')
    all_nodes = set([treatment_col, outcome] + causes)
    nodes_str = "; ".join(f'"{n}"' for n in all_nodes)
    edges_str = "; ".join(edges)
    graph_str = f"digraph {{ {nodes_str}; {edges_str} }}"

    model = CausalModel(
        data=df_est,
        treatment=treatment_col,
        outcome=outcome,
        common_causes=causes if causes else None,
        graph=graph_str,
    )

    identified = model.identify_effect(proceed_when_unidentifiable=True)
    estimate = model.estimate_effect(identified, method_name="backdoor.linear_regression")
    ate = float(estimate.value)

    refute_random = model.refute_estimate(
        identified, estimate,
        method_name="random_common_cause",
        num_simulations=100,
    )

    refute_placebo = model.refute_estimate(
        identified, estimate,
        method_name="placebo_treatment_refuter",
        placebo_type="permute",
        num_simulations=100,
    )

    return {
        "treatment": treatment,
        "outcome": outcome,
        "lag": lag,
        "ate": ate,
        "refute_random_p": float(refute_random.refutation_result.get("p_value", 0)),
        "refute_placebo_p": float(refute_placebo.refutation_result.get("p_value", 0)),
    }

print(f"Estimating causal effects for {len(causal_parents)} parents of '{TARGET}'...\n")
effects = []
for parent in causal_parents:
    try:
        result = estimate_causal_effect(
            df=df_slim,
            treatment=parent["source"],
            outcome=TARGET,
            lag=parent["lag"],
        )
        effects.append(result)
        robust = result["refute_random_p"] > 0.05 and result["refute_placebo_p"] > 0.05
        status = "ROBUST" if robust else "FRAGILE"
        print(f"  [{status}] {result['treatment']} (lag={result['lag']}) -> {TARGET}")
        print(f"           ATE={result['ate']:.4f}, "
              f"random_p={result['refute_random_p']:.3f}, "
              f"placebo_p={result['refute_placebo_p']:.3f}")
    except Exception as e:
        print(f"  [ERROR] {parent['source']} (lag={parent['lag']}): {e}")

df_effects = pd.DataFrame(effects)
print(f"\n{'=' * 60}")
print(f"SUMMARY: {len(effects)} effects estimated, "
      f"{sum(1 for e in effects if e['refute_random_p'] > 0.05 and e['refute_placebo_p'] > 0.05)} robust")


In [ ]:
if effects:
    df_eff = pd.DataFrame(effects)
    df_eff["robust"] = (df_eff["refute_random_p"] > 0.05) & (df_eff["refute_placebo_p"] > 0.05)
    df_eff["label"] = df_eff["treatment"] + " (lag=" + df_eff["lag"].astype(str) + ")"
    df_eff = df_eff.sort_values("ate")

    colors = ["green" if r else "gray" for r in df_eff["robust"]]

    fig = go.Figure(go.Bar(
        x=df_eff["ate"],
        y=df_eff["label"],
        orientation="h",
        marker_color=colors,
        text=[f"ATE={a:.3f}" for a in df_eff["ate"]],
        textposition="outside",
    ))
    fig.update_layout(
        title=f"Causal Effects on {TARGET}<br><sub>Green = passed both refutation tests</sub>",
        xaxis_title="Average Treatment Effect (ATE)",
        yaxis_title="Causal Parent",
        height=max(400, len(effects) * 40),
    )
    fig.show()
else:
    print("No effects to visualize.")


### 3e. Multi-Target: What Drives Each Sleep Dimension?

Run the same causal discovery for secondary targets (sleep efficiency, deep sleep, HRV) to see if different factors drive different aspects of sleep quality.


In [ ]:
all_results = {}

targets_to_analyze = [t for t in [TARGET] + SECONDARY_TARGETS if t in df_slim.columns]

for tgt in targets_to_analyze:
    tgt_idx = var_names.index(tgt)
    parents = []
    for source_idx in range(len(var_names)):
        for lag in range(1, MAX_LAG + 1):
            if graph[source_idx, tgt_idx, lag] == "-->":
                parents.append({
                    "source": var_names[source_idx],
                    "lag": lag,
                    "strength": float(val_matrix[source_idx, tgt_idx, lag]),
                    "p_value": float(p_matrix[source_idx, tgt_idx, lag]),
                })
    parents.sort(key=lambda x: abs(x["strength"]), reverse=True)
    all_results[tgt] = parents

print("CAUSAL DRIVERS BY SLEEP DIMENSION")
print("=" * 60)
for tgt, parents in all_results.items():
    print(f"\n{tgt}:")
    if parents:
        for p in parents[:5]:
            print(f"  {p['source']} (lag={p['lag']}) | strength={p['strength']:.3f}")
    else:
        print("  No significant causal parents found")


In [ ]:
all_causes = set()
for parents in all_results.values():
    for p in parents:
        all_causes.add(p["source"])

cause_list = sorted(all_causes)
target_list = list(all_results.keys())

strength_matrix = np.zeros((len(cause_list), len(target_list)))
for j, tgt in enumerate(target_list):
    for p in all_results[tgt]:
        if p["source"] in cause_list:
            i = cause_list.index(p["source"])
            strength_matrix[i, j] = p["strength"]

if cause_list:
    fig = px.imshow(
        strength_matrix,
        x=target_list,
        y=cause_list,
        color_continuous_scale="RdBu_r",
        zmin=-0.5, zmax=0.5,
        text_auto=".2f",
        title="Causal Strength: What Drives Each Sleep Dimension?",
        labels=dict(x="Sleep Target", y="Causal Parent", color="Strength"),
    )
    fig.update_layout(height=max(400, len(cause_list) * 30), width=600)
    fig.show()
else:
    print("No causal parents found for any target -- try relaxing ALPHA or collecting more data.")


### Summary & Actionable Insights

The analysis above identifies which measurable behaviors **causally drive** your sleep quality -- not just what correlates with it. Key questions answered:

1. **What affects total sleep duration?** -> Check the causal parents of `total_sleep_minutes`
2. **What affects deep sleep specifically?** -> Check `sws_minutes` parents
3. **What affects next-day recovery?** -> Check `hrv_rmssd` and `recovery_score` parents
4. **Are the effects robust?** -> Only green bars in the ATE chart survived refutation

**Interpreting results:**
- A positive ATE means: more of that factor -> more sleep (in minutes)
- Lag = how many days before the effect materializes
- Robust = the effect doesn't vanish when adding random confounders or shuffling the treatment

**Next steps:**
- Collect more data (>90 days) for stronger statistical power
- Add journal entries (alcohol, caffeine, stress) as potential confounders
- Run with stricter alpha (0.01) to reduce false positives


In [ ]:
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "date": str(date.today()),
    "n_days": len(df_slim),
    "variables": var_names,
    "primary_target": TARGET,
    "causal_parents": causal_parents,
    "effects": effects,
    "multi_target_results": {k: v for k, v in all_results.items()},
}

(results_dir / f"causal_analysis_{date.today().isoformat()}.json").write_text(
    json.dumps(summary, indent=2, default=str)
)
print(f"Results saved to {results_dir}/causal_analysis_{date.today().isoformat()}.json")
